# Listener Prior Training (GPU) on Google Colab

This notebook trains the retrieval-based listener prior model on Colab (GPU if available) and saves artifacts to **Google Drive** so they persist.

Artifacts:
- `encoder_best/` (updated whenever MRR@10 improves)
- `encoder_last/`
- `best_eval.json`, `train_progress.json`, `train_summary.json`


## 0) Runtime Settings

In Colab: **Runtime → Change runtime type → GPU**.


In [ ]:
!nvidia-smi || true
import sys
print(sys.version)


## 1) Mount Google Drive

Runs will be written to: `MyDrive/listener_prior_runs/`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2) Get The Repo Into Colab

Option A (recommended): open this notebook from your repo (GitHub → Colab), so the repo is already present.

Option B: set `REPO_URL` and clone.


In [ ]:
import os
import pathlib

# If you're running from a cloned repo already, set PROJECT_DIR='.'
PROJECT_DIR = os.environ.get('PROJECT_DIR', '/content/listener-prior')
REPO_URL = os.environ.get('REPO_URL', '')  # e.g. 'https://github.com/<you>/<repo>.git'

if not os.path.exists(PROJECT_DIR) or not os.path.exists(os.path.join(PROJECT_DIR, 'scripts')):
    if not REPO_URL:
        raise SystemExit('Set REPO_URL env var or set PROJECT_DIR to an existing checkout.')
    !rm -rf "$PROJECT_DIR"
    !git clone --depth 1 "$REPO_URL" "$PROJECT_DIR"

print('PROJECT_DIR:', PROJECT_DIR)
print('Contains scripts/:', os.path.exists(os.path.join(PROJECT_DIR, 'scripts')))
pathlib.Path(PROJECT_DIR).resolve()


## 3) Install Python Dependencies

Colab already ships with CUDA-enabled PyTorch. We avoid reinstalling `torch` to prevent breaking CUDA.


In [ ]:
%cd {PROJECT_DIR}
!python -m pip install -U pip
!grep -v '^torch' requirements.txt > /tmp/requirements_no_torch.txt
!python -m pip install -r /tmp/requirements_no_torch.txt
# Optional (fast retrieval indexing)
!python -m pip install faiss-cpu || true


In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())


## 4) (Optional) Hugging Face Token

If you hit Hugging Face rate limits, set a token. Recommended: use a Colab secret, then export it here.


In [ ]:
import os
# os.environ['HF_TOKEN'] = 'hf_...'
# os.environ['HUGGINGFACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))


## 5) Long Training (6–9 Hours) Using Best Hyperparams

This uses tuned defaults from `scripts/train_long.py` / `src/model.py`.
Artifacts are written **directly to Drive** so they persist across disconnects.


In [ ]:
import os, time
run_id = time.strftime('run_colab_%Y%m%d_%H%M%S')
OUTPUT_DIR = f'/content/drive/MyDrive/listener_prior_runs/{run_id}'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('OUTPUT_DIR:', OUTPUT_DIR)


In [ ]:
# 8h budget (change to any value in [6, 9])
!python scripts/train_long.py   --dataset multiwoz   --output_dir "{OUTPUT_DIR}"   --device auto   --time_budget_hours 8   --max_dialogs 8437   --history_turns 6   --eval_every_steps 5000   --max_eval_examples 2000


## 6) Offline Demo

Runs retrieval + keyword/keyterm extraction from the trained artifacts.


In [ ]:
!python -m src.demo_offline --run "{OUTPUT_DIR}" --topk 5 --device auto
